### LIBRARY IMPORTS

In [4]:
import yaml
import os
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import copy

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegressionCV
from sklearn.metrics import log_loss
from sklearn.preprocessing import LabelEncoder
from sklearn.svm import SVC
from torch.utils.data import DataLoader, TensorDataset

if os.getcwd().endswith('notebooks'):
    os.chdir('..')

from src import *
from src.data_manager import DataManager
from src.processor import Processor
from src.nn_regressor import NNRegressor

### CONFIGURATION

In [12]:
data_manager = DataManager()

datasets_config, modeling_config = data_manager.load_config()

active_dataset = "heart"
active_dataset_config = datasets_config[active_dataset]

problem_type = active_dataset_config["problem_type"]

gradient_boosting_config = modeling_config["gradient_boosting"]

weak_learner_key = gradient_boosting_config["weak_learner_key"]
weak_learner_config = modeling_config[weak_learner_key]

processor = Processor(**active_dataset_config)

train, valid, test = data_manager.load_processed_data()

X_train, y_train = processor.split_features_target(train)
X_valid, y_valid = processor.split_features_target(valid)

y_train, y_valid = processor.transform_target(y_train, y_valid)

### LOGISTIC REGRESSION

In [15]:
logistic = LogisticRegressionCV(Cs=10, cv=5)
logistic.fit(X_train, y_train)

logistic_test_preds = logistic.predict_proba(X_valid)

print('-' * 50)
print(f"Logistic best hyperparameter: {logistic.C_[-1]}")
print(f"Logistic validation cross entropy: {log_loss(y_valid, logistic_test_preds)}")

c:\Users\oriol\Desktop\Projects\TFG\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1780: FutureWarning: The default value for l1_ratios will change from None to (0.0,) in version 1.10. From version 1.10 onwards, only array-like with values in [0, 1] will be allowed, None will be forbidden. To avoid this warning, explicitly set a value, e.g. l1_ratios=(0,).
  warnings.warn(
c:\Users\oriol\Desktop\Projects\TFG\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1823: FutureWarning: The fitted attributes of LogisticRegressionCV will be simplified in scikit-learn 1.10 to remove redundancy. Set`use_legacy_attributes=False` to enable the new behavior now, or set it to `True` to silence this warning during the transition period while keeping the deprecated behavior for the time being. The default value of use_legacy_attributes will change from True to False in scikit-learn 1.10. See the docstring of LogisticRegressionCV for more details.
  warnings.warn(


--------------------------------------------------
Logistic best hyperparameter: 0.3593813663804626
Logistic validation cross entropy: 0.2726516911100664


### SUPPORT VECTOR MACHINE

In [6]:
linear_svm = SVC(kernel='linear', C=1.0, probability=True)
linear_svm.fit(X_train, y_train)

linear_svm_preds = linear_svm.predict_proba(X_test)

rbf_svm = SVC(kernel='rbf', gamma='scale', C=1.0, probability=True)
rbf_svm.fit(X_train, y_train)

rbf_svm_preds = rbf_svm.predict_proba(X_test)

print(f"Linear SVM cross entropy: {log_loss(y_test, linear_svm_preds)}")
print(f"RBF SVM cross entropy: {log_loss(y_test, rbf_svm_preds)}")

Linear SVM cross entropy: 0.27422544802725946
RBF SVM cross entropy: 0.29541181470610334


### RANDOM FOREST

In [36]:
rf = RandomForestClassifier(n_estimators=1_000, max_depth=5, max_features="sqrt")
rf.fit(X_train, y_train)

rf_test_preds = rf.predict_proba(X_test)

print(f"Random forest validation cross entropy: {log_loss(y_test, rf_test_preds)}")

Random forest validation cross entropy: 0.3188188598584369


### NEURAL NETWORK

In [ ]:
class MLPClassifier(NNRegressor):
    def __init__(self):
        super().__init__(epochs=100, learning_rate=0.0001, hidden_size=[32, 16], batch_size=32)

    def fit(
        self,
        X_train: np.ndarray,
        y_train: np.ndarray,
        X_valid: np.ndarray,
        y_valid: np.ndarray, 
        epochs: int = 100,
        patience: int = 10
    ) -> None:
        input_size = X_train.shape[1]
        
        unique_classes = np.unique(y_train)
        output_size = len(unique_classes)
        
        self._get_network(input_size, output_size)
        
        train_loader = self._prepare_loader(X_train, y_train)
        
        X_valid_t = torch.from_numpy(X_valid).to(torch.float32).to(self.device)
        y_valid_t = torch.from_numpy(y_valid).to(torch.long).to(self.device).squeeze()

        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(self.parameters(), lr=self.learning_rate)
        
        best_loss = float('inf')
        best_model = None
        early_stop_count = 0

        for epoch in range(epochs):
            self.train()
            train_loss = 0
            for batch_X, batch_y in train_loader:
                batch_X, batch_y = batch_X.to(self.device), batch_y.to(self.device)
                
                optimizer.zero_grad()
                preds = self(batch_X)
                loss = criterion(preds, batch_y.to(torch.long).to(self.device).squeeze())
                loss.backward()
                optimizer.step()
                train_loss += loss.item()

            self.eval()
            with torch.no_grad():
                val_preds = self(X_valid_t)
                val_loss = criterion(val_preds, y_valid_t).item()

            current_lr = optimizer.param_groups[0]['lr']
            print(f"Epoch {epoch + 1}. Validation cross entropy: {val_loss:.6f}")

            if val_loss < best_loss:
                best_loss = val_loss
                best_model = copy.deepcopy(self.state_dict())
                early_stop_count = 0
            else:
                early_stop_count += 1

            if early_stop_count >= patience:
                print(f"Early stopping on epoch {epoch + 1}!! Best validation MSE: {best_loss:.6f}")
                break
                
        if best_model:
            self.load_state_dict(best_model)

mlp_regressor = MLPClassifier()
mlp_regressor.fit(X_train, y_train, X_valid, y_valid)

Epoch 1. Validation cross entropy: 0.275687
Epoch 2. Validation cross entropy: 0.275167
Epoch 3. Validation cross entropy: 0.272996
Epoch 4. Validation cross entropy: 0.272814
Epoch 5. Validation cross entropy: 0.272720
Epoch 6. Validation cross entropy: 0.272712
Epoch 7. Validation cross entropy: 0.272334
Epoch 8. Validation cross entropy: 0.272469
Epoch 9. Validation cross entropy: 0.272373
Epoch 10. Validation cross entropy: 0.272647
Epoch 11. Validation cross entropy: 0.272118
Epoch 12. Validation cross entropy: 0.272345
Epoch 13. Validation cross entropy: 0.272261
Epoch 14. Validation cross entropy: 0.272305
Epoch 15. Validation cross entropy: 0.273232
Epoch 16. Validation cross entropy: 0.272916
Epoch 17. Validation cross entropy: 0.272310
Epoch 18. Validation cross entropy: 0.273090
Epoch 19. Validation cross entropy: 0.272183
Epoch 20. Validation cross entropy: 0.272253
Epoch 21. Validation cross entropy: 0.272170
Early stopping on epoch 21!! Best validation MSE: 0.272118
